In [ ]:
import sys
import os
from pathlib import Path

if 'google.colab' in sys.modules:
    print("Ambiente Colab rilevato. Inizializzazione in corso...")
    
    # 1. Monta Drive (se non già montato)
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
        
    # Imposta la root del Drive su Colab
    BASE_DIR = Path('/content/drive/MyDrive/Progetto_MLDM')

    GIT_BRANCH = 'refactor-continuo'
    
    # 2. Clona/Aggiorna la repo
    REPO_DIR = '/content/crop-spatial-classification'
    if not os.path.exists(REPO_DIR):
        !git clone -b {GIT_BRANCH} https://github.com/SimoRinaldi/crop-spatial-classification.git {REPO_DIR}
    else:
        !cd {REPO_DIR} && git fetch origin && git checkout {GIT_BRANCH} && git pull origin {GIT_BRANCH}
    
    # 3. Aggancia la cartella per permettere gli import
    if REPO_DIR not in sys.path:
        sys.path.append(REPO_DIR)
        
    # 4. Installa i requisiti in modo silenzioso
    %pip install -q -r {REPO_DIR}/requirements.txt
    print("Setup ambiente Colab completato! Branch attivo su Colab: ", GIT_BRANCH)
    
else:
    print("Ambiente Locale rilevato. Procedo con l'esecuzione...")
    
    BASE_DIR = Path.cwd().parent

    if str(BASE_DIR) not in sys.path:
        sys.path.append(str(BASE_DIR))

# =============================
# DEFINIZIONE PATH UNIVERSALI
# =============================
DATA_DIR = BASE_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
INTERIM_DIR = DATA_DIR / 'interim'
PROCESSED_DIR = DATA_DIR / 'processed'

# Test di verifica per assicurarsi che i dati siano accessibili
if RAW_DIR.exists():
    print(f"✅ Collegamento ai dati riuscito! Cartella raw: {RAW_DIR}")
else:
    print(f"❌ Attenzione: Cartella non trovata in {RAW_DIR}. Verifica il nome, il mount o il path locale.")

In [ ]:
import pandas as pd
from src.test_api2 import download_sentinel_data

# carica i punti estratti nel Notebook 01 (ground truth)
points_file = DATA_DIR / "interim" / "points.json"
df_points = pd.read_json(points_file)

# estrae la lista degli anni associati ai punti
target_years = sorted(df_points["year"].unique().tolist())
print(f"Trovati {len(df_points)} punti da scaricare per gli anni: {target_years}")

# workers in base ai core del PC
workers = min(16, os.cpu_count() or 4)

# cartella per i dati satellitari che verranno scaricati
out_directory = DATA_DIR / "processed" / "sentinel2_data"

download_errors = download_sentinel_data(
    points_df=df_points, 
    years_to_fetch=target_years, 
    max_workers=workers,
    out_dir=str(out_directory),
    zona="capitanata",
    tipo_aggregazione="monthly",
    keep_intermediate_files=True            # impedisce la cancellazione dei dati in caso di crash
)

# report finale
if download_errors:
    print(f"\nDownload completato con errori su {len(download_errors)} punti:")
    for pid, err_list in list(download_errors.items())[:5]: # mostra solo i primi 5 errori
        print(f" - {pid}: {err_list[0]}")
else:
    print(f"\nTutti i {len(df_points)} punti sono stati scaricati con successo in: {out_directory.resolve()}")

In [ ]:
import glob
import json
from pathlib import Path

# Recupera il numero totale di punti/campi attesi da points.json (o da df_points se già in memoria)
points_file = DATA_DIR / "interim" / "points.json"
if "df_points" in globals() and df_points is not None:
    expected_points = len(df_points)
elif points_file.exists():
    with open(points_file, "r", encoding="utf-8") as f:
        expected_points = len(json.load(f))
else:
    expected_points = 0

expected_folders = expected_points

sentinel_dir = DATA_DIR / "processed" / "sentinel2_data"
point_folders = sorted(
    [p for p in sentinel_dir.glob("point_*") if p.is_dir()],
    key=lambda p: int(p.name.split("_")[1]) if p.name.split("_")[1].isdigit() else p.name,
) if sentinel_dir.exists() else []

created_folders = len(point_folders)
total_points = created_folders  # retrocompatibilità con la variabile precedente
completed_points = 0
total_files = 0

for p in point_folders:
    if list(p.glob("sentinel2_data_*.tif")):
        completed_points += 1
    total_files += len(list(p.glob("*.tif")))

# Fallback se points.json non è ancora disponibile
if expected_points == 0:
    expected_points = created_folders
    expected_folders = created_folders

# Calcolo percentuali
pct_folders = (created_folders / expected_folders * 100) if expected_folders > 0 else 0.0
pct_completed = (completed_points / expected_points * 100) if expected_points > 0 else 0.0

print("=" * 50)
print("📊 REPORT DELLO STATO DI DOWNLOAD")
print("=" * 50)
print(f"📁 Cartelle campi create: {created_folders} / {expected_folders} ({pct_folders:.1f}%)")
print(
    f"✅ Campi COMPLETATI al 100%: {completed_points} / {expected_points} ({pct_completed:.1f}%)"
)
print(f"📦 Totale file TIF presenti su Drive: {total_files}")
print("=" * 50)
